# Aprendizaje Estadístico y Data Mining

## Práctica 3: Bosques incendiados

### Objetivo
El SEPRONA quiere saber qué cómo afectan distintas características meteorológicas a la cantidad de bosquequemado en un incendio. Para ello tiene información sobre casos anteriores. La información se encuentra en el archivo **“incendios.csv”** y está compuesta por distintas características de las cuales interesa saber la influencia de la temperatura, humedad, viento y lluvia.

Basándonos en el enunciado, podemos determinar que nos encontramos ante un problema de **regresión**, ya que se pide estudiar como unas ciertas variables (características meteorológicas) afectan al area de un incendio (variable objetivo).

Instalar librerias:

In [188]:
#%pip install pandas scikit-learn numpy matplotlib scipy

Importar librerías necesarias:

In [189]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

**Enunciado:** Crea al menos dos modelos con dos de las variables a estudiar. Haz una interpretación de ambos y di como cambian.  

**Solución**

#### Carga y preprocesamiento del dataset
En primer lugar, comenzaremos cargando el dataset que contiene la información necesaria. Una vez cargado, para evitar que los datos estén demasiado sesgados, aplicaremos una transformación logaritmica (*log-transformación*) en el campo del area, ya que en este tipo de incendios, pueden ser pequeños, lo que tendría un valor entre 0 y 1, o muy grandes, que puede irse a más de 100. Para evitar eso, le aplicamos un logaritmo a este atributo, de tal manera que obtenemos una distribución más simétrica, mucho menos sesgada, y conseguimos relaciones lineales. 

In [190]:
df = pd.read_csv("data/incendios.csv")

df = df.drop(columns=['month', 'day']) # Eliminamos las columnas que no vamos a utilizar

df['area'] = df['area'].replace(0, 0.1) # Cambiamos los 0 por 0.1 para evitar hacer log(0)

df['log_area'] = np.log(df['area'])

Antes de empezar a crear los modelos, estudiaremos la correlación de las diferentes variables con el área.

In [191]:
correlations = {}
for column in df.columns:
    if column != 'area' and column != 'log_area':
        corr, _ = pearsonr(df[column], df['area'])
        correlations[column] = corr

print("---- CORRELACIONES ----")
for feature, corr_value in correlations.items():
    print(f"Correlación {feature} - area: {corr_value:.4f}")

---- CORRELACIONES ----
Correlación X - area: 0.0633
Correlación Y - area: 0.0448
Correlación FFMC - area: 0.0401
Correlación DMC - area: 0.0730
Correlación DC - area: 0.0493
Correlación ISI - area: 0.0082
Correlación temp - area: 0.0978
Correlación RH - area: -0.0755
Correlación wind - area: 0.0123
Correlación rain - area: -0.0074


#### Creación de modelos
Ya con el dataset cargado y preprocesado, podemos proceder con la creación de los modelos de regresión.  

**- Modelo 1:** Para este primer modelo, utilizaremos las variables *temp* y *RH* (Temperatura con humedad relativa en el aire).

In [192]:
X1 = df[['temp', 'RH']]
y = df['log_area']

X_train, X_test, y_train, y_test = train_test_split(X1, y, test_size=0.2, random_state=42)

model1 = LinearRegression()
model1.fit(X_train, y_train)


print("---- COEFICIENTES ----")
print("Coeficientes (temp, RH):", model1.coef_)
print("Intercepto:", model1.intercept_)
print("R2 score en test:", model1.score(X_test, y_test))

---- COEFICIENTES ----
Coeficientes (temp, RH): [0.01633019 0.00037388]
Intercepto: -0.46604610573650845
R2 score en test: 0.006633124764173437


Viendo estos resultados podemos sacar las siguientes conclusiones:
- **Coeficientes**
  - **Coeficiente *temp*** (0.0163): Con este valor, podríamos decir que por cada unidad de temperatura que aumenta, el area del **incendio también aumenta *0.0163 unidades***, es decir, que tiene una relación de directa ligera con respecto al area.  
  
  - **Coeficiente *RH*** (0.0037): En este caso podemos ver que el efecto de la humedad relativa en la extensión del área quemada es muy baja. En este modelo no parece de manera significativa.  

- **Intercepto** (-0.446)
- **R2** (0.0066): El score R2 del modelo es muy bajo, por lo que no predice mucho mejor que la media. Esto puede deberse o bien a la falta de datos de entrenamiento, o bien porque la relación entre las variables con respecto al área sea débil.

**- Modelo 2:** Para este primer modelo, utilizaremos las variables *wind* y *rain*.

In [193]:
X2 = df[['wind', 'rain']]

X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y, test_size=0.2, random_state=42)

model2 = LinearRegression()
model2.fit(X_train2, y_train2)

print("---- COEFICIENTES ----")
print("Coeficientes (wind, rain):", model2.coef_)
print("Intercepto:", model2.intercept_)
print("R2 score en test:", model2.score(X_test2, y_test2))


---- COEFICIENTES ----
Coeficientes (wind, rain): [0.0499293  0.23366332]
Intercepto: -0.345620838081466
R2 score en test: 0.010951565828036691


Las conclusiones que podemos sacar de este segundo modelo son prácticamente iguales que las del primero. Podríamos destacar el coeficiente de la lluvia, ya que el valor que da podría parecer contradictorio, ya que lo más lógico sería que cuanto más llueva, menos probabilidades de incendio. Aún así, podría ocurrir que la humedad del suelo y del ambiente, puedas favorecer de alguna manera la propagación del fuego en caso de un indencio forestal.

**Enunciado:** Realiza otros dos modelos con tres variables. Haz una interpretación de ellos. Selecciona el que creas más útil. Dibújalo si es posible teniendo en cuenta las variables más influyentes.

**Solución**

#### Creación de modelos con 3 variables

In [ ]:
# -------- Modelo 3 --------
features_A = ['temp', 'wind', 'RH']
X_A = df[features_A]
X_A_train, X_A_test, y_train, y_test = train_test_split(X_A, y, test_size=0.2, random_state=42)
model_A = LinearRegression().fit(X_A_train, y_train)
y_pred_A = model_A.predict(X_A_test)
r2_A = r2_score(y_test, y_pred_A)

print("Modelo A (temp, wind, RH):")
print("  Coeficientes:", model_A.coef_)
print("  Intercepto:", model_A.intercept_)
print("  R2:", r2_A)

# -------- Modelo 4 --------
features_B = ['FFMC', 'DMC', 'ISI']
X_B = df[features_B]
X_B_train, X_B_test, y_train, y_test = train_test_split(X_B, y, test_size=0.2, random_state=42)
model_B = LinearRegression().fit(X_B_train, y_train)
y_pred_B = model_B.predict(X_B_test)
r2_B = r2_score(y_test, y_pred_B)

print("\nModelo B (FFMC, DMC, ISI):")
print("  Coeficientes:", model_B.coef_)
print("  Intercepto:", model_B.intercept_)
print("  R2:", r2_B)


Modelo A (temp, wind, RH):
  Coeficientes: [0.02118987 0.06670236 0.00055676]
  Intercepto: -0.8325950501391874
  R2: 0.024412028348051784

Modelo B (FFMC, DMC, ISI):
  Coeficientes: [ 0.03275399  0.00297543 -0.04700595]
  Intercepto: -3.0276781692917196
  R2: -0.02083105944591801
